# Minilink showcase

Minilink is a Python-native block-diagram framework for modeling, simulating,
optimizing, and visualizing dynamical systems. The promise: **the code reads
like the textbook math**, and **everything is a `System`** — plants,
controllers, sources, and full diagrams alike.

This notebook is a guided tour:

1. [Quick start](#1.-Quick-start) — a closed-loop pendulum in a few lines
2. [Everything is a System](#2.-Everything-is-a-System) — ports, parameters, trajectories
3. [Write your own model](#3.-Write-your-own-model) — textbook `f(x, u, t, params)`
4. [Compose diagrams](#4.-Compose-diagrams) — operators, autowire, explicit wiring
5. [Plant catalog](#5.-Plant-catalog) — ready-to-use models with animation geometry
6. [Phase-plane analysis](#6.-Phase-plane-analysis)
7. [Animation renderers](#7.-Animation-renderers) — matplotlib, plotly, meshcat, pygame, interactive
8. [Compiled execution and JAX](#8.-Compiled-execution-and-JAX) — JIT + autodiff
9. [Trajectory optimization](#9.-Trajectory-optimization) — cart-pole swing-up
10. [Symbolic mechanics](#10.-Symbolic-mechanics-(optional)) — SymPy equations of motion
11. [Physics engine](#11.-Physics-engine-(experimental)) — JAX contact dynamics
12. [Where to go next](#12.-Where-to-go-next)

Sections 8–11 degrade gracefully when the optional extras (`jax`, `sympy`)
are not installed.

**Run it locally** with minilink installed (`pip install -e .` from the repo
root), or uncomment the Colab setup cell below.

In [ ]:
# Setup for Google Colab (uncomment to run there):
# !git clone https://github.com/alx87grd/minilink
# %cd minilink
# %pip install -e .

## 1. Quick start

A pendulum plant and a PD controller, composed into a closed loop with the
`@` operator. The diagram is itself a `System`: simulate it, plot it,
animate it.

In [ ]:
import numpy as np

from minilink.control.linear import PDController
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum

controller = PDController()
plant = Pendulum()

plant.x0[0] = 2.0        # initial angle [rad]
plant.params["l"] = 5.0  # arm length [m]
plant.params["m"] = 1.0  # mass [kg]

controller.params["kp"] = 10.0
controller.params["kd"] = 5.0

diagram = controller @ plant

traj = diagram.compute_trajectory(tf=10.0, n_steps=1001)
diagram.plot_trajectory()

In [ ]:
# The wiring topology (requires a system Graphviz install)
diagram.plot_diagram()

In [ ]:
# Inline animation (auto-selects HTML output in notebooks)
diagram.animate()

## 2. Everything is a System

A `System` owns its equations (`f`, `h`), named input/output **ports** with
labels, units, and bounds, default **parameters**, and a default initial
condition. It does *not* hide simulation state — trajectories live in the
returned `Trajectory` object.

In [ ]:
print("Plant:   ", plant.name)
print("states:  ", plant.n, plant.state.labels, plant.state.units)
print("inputs:  ", plant.m, list(plant.inputs))
print("outputs: ", list(plant.outputs))
print("params:  ", plant.params)

In [ ]:
# The closed-loop diagram is a System too: it exposes flattened states and
# its own boundary ports, so it can be embedded in a larger diagram.
print("Diagram states:", diagram.n, " inputs:", list(diagram.inputs))
print("Trajectory:    ", traj.x.shape, "states x samples")

## 3. Write your own model

Subclass `DynamicSystem` and implement `f(x, u, t, params)`. The equation
code stays textbook-style: unpack the parameters, name the signals, write
the ODE. Use object defaults only when `params is None`.

In [ ]:
from minilink.blocks.sources import Step
from minilink.core.system import DynamicSystem


class MassSpringDamper(DynamicSystem):
    """m x'' + c x' + k x = u"""

    def __init__(self):
        super().__init__(n=2, input_dim=1, expose_state=True)
        self.params = {"m": 1.0, "k": 4.0, "c": 0.3}
        self.state.labels = ["x", "dx"]

    def f(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        m, k, c = params["m"], params["k"], params["c"]

        position, velocity = x
        force = u[0]

        dx = np.zeros(2)
        dx[0] = velocity
        dx[1] = (force - c * velocity - k * position) / m
        return dx


msd = MassSpringDamper()
msd.x0[0] = 1.0

step = Step(final_value=np.array([10.0]), step_time=2.0)
chain = step >> msd

chain.compute_trajectory(tf=20.0, n_steps=1001)
chain.plot_trajectory(signals=("x", "step:y"))

## 4. Compose diagrams

| Shortcut | Meaning |
| --- | --- |
| `a + b + c` | add subsystems without wiring |
| `source >> plant` | chain output to input |
| `controller @ plant` | simple feedback diagram |
| `.autowire(strict=True)` | connect matching named ports |

Shortcuts flatten their operands (no nested diagrams), and any internal
signal can be plotted by `"subsystem_id:port_id"` name.

In [ ]:
loop = (Step(final_value=[1.0]) + PDController() + Pendulum()).autowire(strict=True)

loop.compute_trajectory(tf=10.0, n_steps=1001)
loop.plot_trajectory(signals=("pendulum:y", "pd_controller:u", "step:y"))

In [ ]:
loop.plot_diagram()

In [ ]:
# The explicit API is always available when shortcuts are too implicit:
from minilink.core.diagram import DiagramSystem

explicit = DiagramSystem()
explicit.add_subsystem(Step(final_value=[1.0]), "step")
explicit.add_subsystem(PDController(), "controller")
explicit.add_subsystem(Pendulum(), "plant")
explicit.connect("step", "y", "controller", "r")
explicit.connect("plant", "y", "controller", "y")
explicit.connect("controller", "u", "plant", "u")
explicit

## 5. Plant catalog

`minilink.dynamics.catalog` ships ready-to-use plants — pendulums,
cart-poles, manipulators, ground vehicles, drones, boats, rockets,
mass-spring-damper chains — each with parameters, labeled ports, and
animation geometry.

In [ ]:
from minilink.dynamics.catalog.aerial.drone import Drone2D
from minilink.dynamics.catalog.pendulum.cartpole import CartPole
from minilink.dynamics.catalog.vehicles.steering import KinematicCar

for sys in (CartPole(), KinematicCar(), Drone2D()):
    print(f"{sys.name:<22} n={sys.n}  m={sys.m}  states={sys.state.labels}")

In [ ]:
# Forced response: drive the cart with a sinusoidal force and watch the pole.
cartpole = CartPole()
cartpole_traj = cartpole.compute_forced(
    lambda t: np.array([3.0 * np.sin(2.0 * t)]),
    tf=8.0,
    n_steps=801,
)
cartpole.plot_trajectory()

In [ ]:
cartpole.animate()

## 6. Phase-plane analysis

`plot_phase_plane()` draws the vector field of any two-state system (or a
2-D slice of a larger one), with optional streamlines and overlaid
trajectories.

In [ ]:
pendulum = Pendulum()
pendulum.state.lower_bound = np.array([-2.0 * np.pi, -6.0])
pendulum.state.upper_bound = np.array([2.0 * np.pi, 6.0])
pendulum.x0 = np.array([-np.pi, 4.0])

free_swing = pendulum.compute_trajectory(tf=5.0, n_steps=501, show=False)
pendulum.plot_phase_plane(free_swing)

## 7. Animation renderers

`animate()` plays a trajectory through swappable renderers:

- `renderer="matplotlib"` (default) — inline HTML in notebooks, native window in scripts
- `renderer="plotly"` — interactive inline animation
- `renderer="meshcat"` — 3D in the browser (great for vehicles and arms)
- `renderer="pygame"` — fast native 2D window

`game()` runs the model **interactively** with keyboard input — useful for
building intuition about a plant before designing a controller. These need a
display, so they are left commented here:

In [ ]:
# pendulum.animate(renderer="meshcat")     # 3D in the browser
# pendulum.game(renderer="pygame")         # drive it with the keyboard
#
# from minilink.dynamics.catalog.vehicles.dynamic_bicycle import DynamicBicycleCar3D
# DynamicBicycleCar3D().game(renderer="meshcat")

## 8. Compiled execution and JAX

Wired diagrams compile into a **flat execution plan**. The NumPy backend
removes the recursive port-resolution overhead of `diagram.f`; the JAX
backend JIT-compiles the flattened dynamics and keeps them traceable for
autodiff.

In [ ]:
import time

evaluator = diagram.compile(backend="numpy")

x = np.zeros(diagram.n)
u = np.zeros(diagram.m)

n_iters = 2000
t0 = time.perf_counter()
for _ in range(n_iters):
    diagram.f(x, u)
t_python = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(n_iters):
    evaluator.f(x, u)
t_numpy = time.perf_counter() - t0

print(f"recursive diagram.f : {1e6 * t_python / n_iters:6.1f} us/call")
print(f"compiled  (numpy)   : {1e6 * t_numpy / n_iters:6.1f} us/call")

Plain NumPy models compile on the NumPy backend. For JAX, equations must be
traceable: basic blocks achieve this with the one-line `xp = array_module(x)`
idiom, and catalog plants ship JAX twins (`JaxCartPole`,
`JaxMechanicalSystem` subclasses) verified against their NumPy reference.

In [ ]:
try:
    import jax

    HAS_JAX = True
except ImportError:
    HAS_JAX = False
    print("JAX not installed - skipping the JAX sections.")

if HAS_JAX:
    from minilink.dynamics.catalog.pendulum.cartpole import JaxCartPole

    jax_sys = JaxCartPole()
    jax_eval = jax_sys.compile(backend="jax")

    xc = np.zeros(jax_sys.n)
    uc = np.zeros(jax_sys.m)

    jax_eval.f(xc, uc, 0.0)  # warm up the JIT cache
    t0 = time.perf_counter()
    for _ in range(n_iters):
        jax_eval.f(xc, uc, 0.0)
    t_jax = time.perf_counter() - t0

    t0 = time.perf_counter()
    for _ in range(n_iters):
        jax_sys.f(xc, uc, 0.0)
    t_pure = time.perf_counter() - t0

    print(f"JaxCartPole.f (interpreted) : {1e6 * t_pure / n_iters:6.1f} us/call")
    print(f"compiled (jax jit)          : {1e6 * t_jax / n_iters:6.1f} us/call")

In [ ]:
if HAS_JAX:
    # The compiled dynamics stay traceable: exact linearization for free.
    A = jax.jacfwd(lambda x: jax_eval.f(x, uc, 0.0))(xc)
    print("df/dx at the upright equilibrium:\n", np.asarray(A).round(3))

## 9. Trajectory optimization

A `PlanningProblem` bundles a system, boundary conditions, and a cost.
Pluggable **transcriptions** (direct collocation, single/multiple shooting)
turn it into a nonlinear program solved by SciPy or Ipopt — optionally with
JAX-exact gradients. Here: a cart-pole swing-up.

In [ ]:
from minilink.core.costs import QuadraticCost
from minilink.planning.problems import PlanningProblem
from minilink.planning.trajectory_optimization.direct_collocation import (
    DirectCollocationOptions,
    DirectCollocationTranscription,
)
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationOptions,
    TrajectoryOptimizationPlanner,
)

swingup = CartPole()
swingup.inputs["u"].lower_bound[0] = -10.0
swingup.inputs["u"].upper_bound[0] = 10.0

x_start = np.array([-2.0, 1.0, 0.0, 0.0])
x_goal = np.array([0.0, np.pi, 0.0, 0.0])  # pole up, cart at the origin

problem = PlanningProblem(
    sys=swingup,
    x_start=x_start,
    x_goal=x_goal,
    cost=QuadraticCost.from_system(
        swingup, Q=np.diag([1.0, 1.0, 0.0, 0.0]), xbar=x_goal
    ),
)

planner = TrajectoryOptimizationPlanner(
    problem,
    transcription=DirectCollocationTranscription(
        DirectCollocationOptions(tf=5.0, n_steps=30)
    ),
    options=TrajectoryOptimizationOptions(
        solve_disp=True,
        optimizer_options={"maxiter": 500, "ftol": 1e-2},
    ),
)

swingup_traj = planner.compute_solution()

In [ ]:
planner.plot_solution(signals=("x", "u"))

In [ ]:
swingup.animate(swingup_traj)

## 10. Symbolic mechanics (optional)

`minilink.symbolic.mechanics` (experimental, needs `sympy`) derives
manipulator-form equations of motion from a DH-chain description —
Lagrange or Kane — and exports the result as a regular minilink mechanical
system, including a JAX-traceable variant.

In [ ]:
try:
    import sympy  # noqa: F401

    HAS_SYMPY = True
except ImportError:
    HAS_SYMPY = False
    print("SymPy not installed - skipping the symbolic section.")

if HAS_SYMPY:
    from minilink.symbolic.mechanics.model import MechanicalModel

    model = MechanicalModel("DoublePendulum")
    n_links = 2

    lengths, com_offsets, masses, inertias = [], [], [], []
    for i in range(1, n_links + 1):
        li, lci = model.parameters(f"l{i} lc{i}")
        lengths.append(li)
        com_offsets.append(lci)
        masses.append(model.parameters(f"m{i}"))
        inertias.append(model.parameters(f"I{i}"))
    g_sym = model.parameters("g")

    coords = model.coordinates(" ".join(f"q{i}" for i in range(1, n_links + 1)))
    dh_table = [
        {"theta": coords[i], "d": 0, "a": lengths[i], "alpha": 0}
        for i in range(n_links)
    ]
    link_properties = [
        {
            "mass": masses[i],
            "inertia": {"Izz": inertias[i]},
            "com_offset": com_offsets[i],
        }
        for i in range(n_links)
    ]

    model.add_dh_chain(dh_table, link_properties)
    model.add_gravity(-g_sym * model.N.y)

    sym_sys = model.derive(method="lagrange", simplify=True)
    display(sym_sys.H)  # symbolic inertia matrix H(q)

In [ ]:
if HAS_SYMPY:
    # Bind numbers to the symbols and export a simulatable minilink system.
    params = {g_sym: 9.81}
    for i in range(n_links):
        params[lengths[i]] = 1.0
        params[com_offsets[i]] = 0.5
        params[masses[i]] = 1.0
        params[inertias[i]] = 0.1

    exported = sym_sys.to_minilink(parameters=params)
    exported.x0 = np.array([0.5, 0.0, 0.0, 0.0])
    exported.compute_trajectory(tf=5.0, n_steps=501)
    exported.plot_trajectory()

## 11. Physics engine (experimental)

`minilink.dynamics.engines` wraps a JAX rigid-body contact engine as one
`DynamicSystem`: spheres, a ground plane, compliant contact — compiled,
differentiable, and usable inside any diagram. Math is TRL 1 (not yet
QA-validated); see `ROADMAP.md`.

In [ ]:
if HAS_JAX:
    from minilink.dynamics.engines.contact_jax import (
        PlaneModel,
        SphereModel,
        make_world_model,
    )
    from minilink.dynamics.engines.world import PhysicsWorldSystem

    # A 3x3 grid of spheres dropped from varied heights onto a plane.
    n_side = 3
    spheres = [SphereModel(mass=1.0, radius=0.25) for _ in range(n_side**2)]
    world = make_world_model(
        spheres,
        PlaneModel(normal=(0.0, 0.0, 1.0), offset=0.0),
        gravity=(0.0, 0.0, -9.81),
        k_contact=1000.0,
        c_contact=1.0,
    )
    bouncy = PhysicsWorldSystem(world, name="SphereGrid")

    # State layout per body: [p(3), q(4), v(3), w(3)].
    x0 = np.zeros(bouncy.n)
    grid = np.linspace(-1.0, 1.0, n_side)
    for i in range(bouncy.world.n_bodies):
        base = 13 * i
        x0[base : base + 3] = [grid[i % n_side], grid[i // n_side], 1.0 + 0.5 * i]
        x0[base + 3 : base + 7] = [1.0, 0.0, 0.0, 0.0]  # unit quaternion
    bouncy.x0 = x0

    bounce_traj = bouncy.compute_trajectory(
        tf=2.0, dt=0.01, solver="rk4_fixedsteps", compile_backend="jax", show=False
    )

    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 3))
    for i in range(bouncy.world.n_bodies):
        ax.plot(bounce_traj.t, bounce_traj.x[13 * i + 2, :])
    ax.set_xlabel("t [s]")
    ax.set_ylabel("sphere height z [m]")
    ax.set_title("9 spheres bouncing on a plane (JAX contact engine)")
    plt.show()

    # 3D playback in the browser:
    # bouncy.animate(renderer="meshcat")

## 12. Where to go next

- **`examples/scripts/`** — runnable demos by topic: diagrams, plots,
  animation, trajectory optimization, symbolic mechanics, physics engine.
- **[README](../../README.md)** — feature overview, install, and the minimal
  call chains for debugging and extending workflows.
- **[DESIGN.md](../../DESIGN.md)** — architecture principles and contracts.
- **[ROADMAP.md](../../ROADMAP.md)** — subsystem maturity and priorities.

Happy modeling!